In [ ]:
import os, shutil, pandas as pd

base_dir = r"C:\Users\HP\CropStressDetection\data\audio"
metadata_path = os.path.join(base_dir, "metadata", "UrbanSound8K.csv")

print("📄 Checking metadata file at:", metadata_path)
assert os.path.exists(metadata_path), "❌ Metadata file not found!"

df = pd.read_csv(metadata_path)
print("✅ Metadata loaded! Total records:", len(df))

# Class grouping
healthy_classes = ['children_playing', 'street_music', 'dog_bark', 'car_horn', 'siren']
stressed_classes = ['air_conditioner', 'engine_idling', 'drilling', 'jackhammer']

healthy_dir = os.path.join(base_dir, "healthy")
stressed_dir = os.path.join(base_dir, "stressed")
os.makedirs(healthy_dir, exist_ok=True)
os.makedirs(stressed_dir, exist_ok=True)

print("📦 Copying files...")
copied = 0

for _, row in df.iterrows():
    src = os.path.join(base_dir, f"fold{row['fold']}", row['slice_file_name'])
    if not os.path.exists(src):
        continue
    if row['class'] in healthy_classes:
        dest = os.path.join(healthy_dir, row['slice_file_name'])
    elif row['class'] in stressed_classes:
        dest = os.path.join(stressed_dir, row['slice_file_name'])
    else:
        continue
    shutil.copy(src, dest)
    copied += 1
    if copied % 500 == 0:
        print(f"  ➜ Copied {copied} files so far...")

print("✅ Files copied successfully!")
print("Healthy samples:", len(os.listdir(healthy_dir)))
print("Stressed samples:", len(os.listdir(stressed_dir)))


📄 Checking metadata file at: C:\Users\HP\CropStressDetection\data\audio\metadata\UrbanSound8K.csv
✅ Metadata loaded! Total records: 8732
📦 Copying files...
  ➜ Copied 500 files so far...
  ➜ Copied 1000 files so far...
  ➜ Copied 1500 files so far...
  ➜ Copied 2000 files so far...
  ➜ Copied 2500 files so far...
  ➜ Copied 3000 files so far...


In [1]:
import os

healthy_path = r"C:\Users\HP\CropStressDetection\data\audio\healthy"
stressed_path = r"C:\Users\HP\CropStressDetection\data\audio\stressed"

print("Healthy path exists:", os.path.exists(healthy_path))
print("Stressed path exists:", os.path.exists(stressed_path))


Healthy path exists: True
Stressed path exists: True


In [3]:
print("Healthy folder contents:", os.listdir(healthy_path)[:5])
print("Stressed folder contents:", os.listdir(stressed_path)[:5])

Healthy folder contents: ['100032-3-0-0.wav', '100263-2-0-117.wav', '100263-2-0-121.wav', '100263-2-0-126.wav', '100263-2-0-137.wav']
Stressed folder contents: ['100852-0-0-0.wav', '100852-0-0-1.wav', '100852-0-0-10.wav', '100852-0-0-11.wav', '100852-0-0-12.wav']


In [4]:
healthy_files = len(os.listdir(healthy_path))
stressed_files = len(os.listdir(stressed_path))

print(f"🌿 Healthy samples: {healthy_files}")
print(f"⚠️ Stressed samples: {stressed_files}")


🌿 Healthy samples: 4358
⚠️ Stressed samples: 4000


In [6]:
import librosa
import numpy as np
import pandas as pd

def extract_features(file_path):
    y, sr = librosa.load(file_path, duration=3, offset=0.5)
    mfccs = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40).T, axis=0)
    return mfccs

features = []
labels = []

# Healthy
for file in os.listdir(healthy_path):
    path = os.path.join(healthy_path, file)
    data = extract_features(path)
    features.append(data)
    labels.append("healthy")

# Stressed
for file in os.listdir(stressed_path):
    path = os.path.join(stressed_path, file)
    data = extract_features(path)
    features.append(data)
    labels.append("stressed")

df = pd.DataFrame(features)
df["label"] = labels

print("✅ Features extracted successfully!")
print(df.head())


C:\Users\HP\anaconda3\lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated
  "class": algorithms.Blowfish,
C:\Users\HP\AppData\Local\Temp\ipykernel_13168\402689018.py:6: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(file_path, duration=3, offset=0.5)
C:\Users\HP\anaconda3\lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
C:\Users\HP\anaconda3\lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=0
  warnings.warn(
C:\Users\HP\AppData\Local\Temp\ipykernel_13168\402689018.py:6: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(file_path, duration=3, offset=0.5)
C:\Users\HP\anaconda3\lib\site-packages\librosa\core\audio

NoBackendError: 

In [7]:
import os
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm

# ---------- PATHS ----------
healthy_path = r"C:\Users\HP\CropStressDetection\data\audio\healthy"
stressed_path = r"C:\Users\HP\CropStressDetection\data\audio\stressed"
save_path = r"C:\Users\HP\CropStressDetection\data\audio_features.csv"

# ---------- VERIFY FOLDERS ----------
if not os.path.exists(healthy_path) or not os.path.exists(stressed_path):
    raise FileNotFoundError("❌ One of the audio folders is missing. Check your paths!")

# ---------- FUNCTION TO EXTRACT FEATURES ----------
def extract_features(file_path):
    try:
        y, sr = librosa.load(file_path, sr=None)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        mfcc_mean = np.mean(mfcc, axis=1)
        return mfcc_mean
    except Exception as e:
        print(f"⚠️ Error processing {file_path}: {e}")
        return None

# ---------- PROCESS ALL FILES ----------
data = []
print("\n🎧 Extracting features from audio files...\n")

for label, folder in [("healthy", healthy_path), ("stressed", stressed_path)]:
    files = os.listdir(folder)
    for file in tqdm(files, desc=f"Processing {label}"):
        if file.endswith(".wav"):
            file_path = os.path.join(folder, file)
            features = extract_features(file_path)
            if features is not None:
                row = list(features) + [label]
                data.append(row)

# ---------- SAVE FEATURES ----------
columns = [f"mfcc_{i}" for i in range(13)] + ["label"]
df = pd.DataFrame(data, columns=columns)
df.to_csv(save_path, index=False)

print(f"\n✅ Feature extraction complete! Saved to:\n{save_path}")
print(f"Total samples processed: {len(df)}")
print(df.head())



🎧 Extracting features from audio files...



Processing healthy:  13%|█▎        | 546/4358 [00:34<05:16, 12.04it/s]C:\Users\HP\AppData\Local\Temp\ipykernel_13168\214322529.py:19: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(file_path, sr=None)
Processing healthy:  13%|█▎        | 550/4358 [00:34<05:05, 12.47it/s]

⚠️ Error processing C:\Users\HP\CropStressDetection\data\audio\healthy\128240-3-0-42.wav: 


Processing stressed:  31%|███       | 1228/4000 [02:00<05:21,  8.62it/s]C:\Users\HP\anaconda3\lib\site-packages\librosa\feature\spectral.py:2148: UserWarning: Empty filters detected in mel frequency basis. Some channels will produce empty responses. Try increasing your sampling rate (and fmax) or reducing n_mels.
  mel_basis = filters.mel(sr=sr, n_fft=n_fft, **kwargs)
Processing stressed: 100%|██████████| 4000/4000 [13:32<00:00,  4.92it/s]  



✅ Feature extraction complete! Saved to:
C:\Users\HP\CropStressDetection\data\audio_features.csv
Total samples processed: 8357
       mfcc_0      mfcc_1     mfcc_2     mfcc_3     mfcc_4     mfcc_5  \
0 -275.918427  119.492798 -98.211777 -66.515129 -42.606045   0.505065   
1 -500.908386  185.106415 -86.532822  49.858849   9.230822  22.548956   
2 -531.195312  186.939941 -70.349167  40.429245   9.121046  18.398588   
3 -476.784424  160.333282 -62.952843  50.751171  -0.174330  32.791603   
4 -521.244690  185.392654 -81.950478  46.473549  11.872088  23.491444   

      mfcc_6     mfcc_7    mfcc_8    mfcc_9    mfcc_10    mfcc_11   mfcc_12  \
0 -28.330935  -5.746867  9.992785  4.795412  15.461892  -0.069880 -2.842674   
1  -3.567174  12.220052  7.720082 -6.460391  16.995657  -6.625117  1.469779   
2   6.283282  15.504061  9.613501 -7.113610  16.179823  -5.710522 -0.899251   
3 -17.469801  24.755478 -3.847783 -1.761176  14.020370 -11.238716  6.290465   
4   4.261836  13.637699  6.093238 -2.8